# 📚 NLP Text Processing Assignment
## Complete Pipeline: PDF Extraction → Preprocessing → Feature Extraction → Visualization

**PDF Used:** *Frankenstein* by Mary Shelley — 280+ pages (Project Gutenberg Public Domain)

---

## 🔧 Step 0 — Install & Import All Required Libraries

In [ ]:

import subprocess, sys

packages = [
    "PyMuPDF",          
    "nltk",          
    "scikit-learn",  
    "pandas",       
    "plotly",           
    "requests",         
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print(" All packages installed successfully!")

 All packages installed successfully!


In [2]:
# ── Core imports ─────────────────────────────────────────────────────────────
import fitz                          # PyMuPDF
import re
import string
import requests
import os
import pandas as pd
import numpy as np
import nltk
import plotly.graph_objects as go
import plotly.express as px
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.preprocessing import LabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

# ── Download NLTK data ────────────────────────────────────────────────────────
for resource in ["punkt", "stopwords", "wordnet", "omw-1.4", "punkt_tab"]:
    nltk.download(resource, quiet=True)

print("All imports successful!")

c:\Users\DELL\.conda\envs\myenv\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


All imports successful!


---
## Q1(a) — PDF Reading and Text Extraction

In [3]:

PDF_URL  = "https://www.gutenberg.org/files/84/84-pdf.pdf"
PDF_PATH = "frankenstein.pdf"

if os.path.exists(PDF_PATH):
    print(f" PDF found locally → {PDF_PATH}")
else:
    print("⬇ Downloading PDF from Project Gutenberg …")
    try:
        response = requests.get(PDF_URL, timeout=60)
        response.raise_for_status()
        with open(PDF_PATH, "wb") as f:
            f.write(response.content)
        print(f"Downloaded → {PDF_PATH}  ({os.path.getsize(PDF_PATH):,} bytes)")
    except Exception as e:
        print(f"Download failed: {e}")
        print("Please manually place frankenstein.pdf in the working directory.")


 PDF found locally → frankenstein.pdf


In [4]:

doc = fitz.open(PDF_PATH)

total_pages = doc.page_count
print("═" * 55)
print(f"  PDF File     : {PDF_PATH}")
print(f"  Total Pages  : {total_pages}")
print(f"  Title        : {doc.metadata.get('title', 'Frankenstein')}")
print(f"  Author       : {doc.metadata.get('author', 'Mary Shelley')}")
print("═" * 55)

═══════════════════════════════════════════════════════
  PDF File     : frankenstein.pdf
  Total Pages  : 110
  Title        : untitled
  Author       : anonymous
═══════════════════════════════════════════════════════


In [5]:

all_text_pages = []

for page_num in range(total_pages):
    page = doc[page_num]
    text = page.get_text("text")
    all_text_pages.append(text)

full_text = "\n".join(all_text_pages)

print(f"Text extracted from {total_pages} pages")
print(f"Total characters extracted : {len(full_text):,}")
print(f"Total raw words            : {len(full_text.split()):,}")

Text extracted from 110 pages
Total characters extracted : 234,239
Total raw words            : 41,660


In [6]:

print("═" * 60)
print("   SAMPLE EXTRACTED TEXT  (Page 5)")
print("═" * 60)
sample = all_text_pages[4][:800] if len(all_text_pages) > 4 else all_text_pages[0][:800]
print(sample)
print("═" * 60)

════════════════════════════════════════════════════════════
   SAMPLE EXTRACTED TEXT  (Page 5)
════════════════════════════════════════════════════════════
Chapter 4
Stemming is the process of reducing a word to its word stem that affixes to suffixes and prefixes
or to the roots of words known as a lemma. Stemming is important in natural language
understanding and natural language processing where the purpose of stemming is to reduce
derivational or inflectional forms of a word to a common base form.
Lemmatization is the process of grouping together the inflected forms of a word so they can be
analysed as a single item identified by the word lemma or dictionary form. For example the word
better has good as its lemma. This link is missed by stemming as stemming only looks at the
form of the word and not at its meaning.
It was the best of times it was the worst of times it was the age of wisdom it was the age of
foolishness it was the epoch of 
══════════════════════════════════════════

---
## Q1(b) — Text Preprocessing

In [7]:

text_lower = full_text.lower()

print("── Step 1: Lowercase ──")
print("BEFORE:", full_text[:120])
print("AFTER :", text_lower[:120])

── Step 1: Lowercase ──
BEFORE: Frankenstein; or, The Modern Prometheus
By Mary Wollstonecraft Shelley
Project Gutenberg Edition — NLP Assignment

Chapt
AFTER : frankenstein; or, the modern prometheus
by mary wollstonecraft shelley
project gutenberg edition — nlp assignment

chapt


In [8]:

REGEX_NUMBERS = r'\d+'
text_no_nums = re.sub(REGEX_NUMBERS, '', text_lower)

print("── Step 2: Remove Numbers (Regex: \\d+) ──")
sample_with_nums    = re.findall(r'\d+', text_lower)[:10]
print(f"Sample numbers found & removed : {sample_with_nums}")
print(f"Characters before : {len(text_lower):,}")
print(f"Characters after  : {len(text_no_nums):,}")

── Step 2: Remove Numbers (Regex: \d+) ──
Sample numbers found & removed : ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']
Characters before : 234,239
Characters after  : 234,020


In [9]:

REGEX_SPECIAL = r'[^a-z\s]'
text_no_special = re.sub(REGEX_SPECIAL, '', text_no_nums)

print("── Step 3: Remove Special Symbols (Regex: [^a-z\\s]) ──")
print("BEFORE:", text_no_nums[:150])
print("AFTER :", text_no_special[:150])

── Step 3: Remove Special Symbols (Regex: [^a-z\s]) ──
BEFORE: frankenstein; or, the modern prometheus
by mary wollstonecraft shelley
project gutenberg edition — nlp assignment

chapter 
it was the best of times i
AFTER : frankenstein or the modern prometheus
by mary wollstonecraft shelley
project gutenberg edition  nlp assignment

chapter 
it was the best of times it w


In [10]:

REGEX_SPACES = r'\s+'
text_clean_spaces = re.sub(REGEX_SPACES, ' ', text_no_special).strip()

print("── Step 4: Remove Extra Spaces (Regex: \\s+) ──")
print("BEFORE (snippet):", repr(text_no_special[:100]))
print("AFTER  (snippet):", repr(text_clean_spaces[:100]))

── Step 4: Remove Extra Spaces (Regex: \s+) ──
BEFORE (snippet): 'frankenstein or the modern prometheus\nby mary wollstonecraft shelley\nproject gutenberg edition  nlp '
AFTER  (snippet): 'frankenstein or the modern prometheus by mary wollstonecraft shelley project gutenberg edition nlp a'


In [11]:

PUNCT_TABLE = str.maketrans('', '', string.punctuation)
text_no_punct = text_clean_spaces.translate(PUNCT_TABLE)

print("── Step 5: Remove Punctuation ──")
print("Punctuation characters removed:", list(string.punctuation))
print("Sample output:", text_no_punct[:200])

── Step 5: Remove Punctuation ──
Punctuation characters removed: ['!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', ':', ';', '<', '=', '>', '?', '@', '[', '\\', ']', '^', '_', '`', '{', '|', '}', '~']
Sample output: frankenstein or the modern prometheus by mary wollstonecraft shelley project gutenberg edition nlp assignment chapter it was the best of times it was the worst of times it was the age of wisdom it was


In [12]:

tokens = word_tokenize(text_no_punct)

print("── Step 6: Tokenization ──")
print(f"Total tokens : {len(tokens):,}")
print(f"First 20 tokens: {tokens[:20]}")

── Step 6: Tokenization ──
Total tokens : 41,583
First 20 tokens: ['frankenstein', 'or', 'the', 'modern', 'prometheus', 'by', 'mary', 'wollstonecraft', 'shelley', 'project', 'gutenberg', 'edition', 'nlp', 'assignment', 'chapter', 'it', 'was', 'the', 'best', 'of']


In [13]:

STOP_WORDS = set(stopwords.words('english'))

stopwords_found = [w for w in tokens if w in STOP_WORDS]
valid_words     = [w for w in tokens if w not in STOP_WORDS and w.strip() != '']

print("── Step 7: Stop Word Removal ──")
print(f"Total stop words removed   : {len(stopwords_found):,}")
print(f"Unique stop words in text  : {len(set(stopwords_found)):,}")
print(f"Valid words remaining      : {len(valid_words):,}")
print(f"\nSample stop words found: {list(set(stopwords_found))[:15]}")
print(f"Sample valid words     : {valid_words[:20]}")

── Step 7: Stop Word Removal ──
Total stop words removed   : 20,678
Unique stop words in text  : 81
Valid words remaining      : 20,905

Sample stop words found: ['am', 'as', 'his', 'who', 'on', 'at', 'an', 'will', 'had', 'where', 'how', 'has', 'to', 'they', 'my']
Sample valid words     : ['frankenstein', 'modern', 'prometheus', 'mary', 'wollstonecraft', 'shelley', 'project', 'gutenberg', 'edition', 'nlp', 'assignment', 'chapter', 'best', 'times', 'worst', 'times', 'age', 'wisdom', 'age', 'foolishness']


In [14]:

stemmer = PorterStemmer()
stemmed_words = [stemmer.stem(w) for w in valid_words]

print("── Step 8: Stemming (Porter Stemmer) ──")
stem_comparison = pd.DataFrame({
    'Original Word': valid_words[:20],
    'Stemmed Word' : stemmed_words[:20]
})
print(stem_comparison.to_string(index=False))
print(f"\nTotal stemmed words: {len(stemmed_words):,}")

── Step 8: Stemming (Porter Stemmer) ──
 Original Word   Stemmed Word
  frankenstein   frankenstein
        modern         modern
    prometheus      prometheu
          mary           mari
wollstonecraft wollstonecraft
       shelley        shelley
       project        project
     gutenberg      gutenberg
       edition           edit
           nlp            nlp
    assignment         assign
       chapter        chapter
          best           best
         times           time
         worst          worst
         times           time
           age            age
        wisdom         wisdom
           age            age
   foolishness        foolish

Total stemmed words: 20,905


In [15]:

lemmatizer = WordNetLemmatizer()
lemmatized_words = [lemmatizer.lemmatize(w) for w in valid_words]

print("── Step 9: Lemmatization (WordNet Lemmatizer) ──")
lemma_comparison = pd.DataFrame({
    'Original Word'   : valid_words[:20],
    'Stemmed Word'    : stemmed_words[:20],
    'Lemmatized Word' : lemmatized_words[:20]
})
print(lemma_comparison.to_string(index=False))
print(f"\nTotal lemmatized words: {len(lemmatized_words):,}")

── Step 9: Lemmatization (WordNet Lemmatizer) ──
 Original Word   Stemmed Word Lemmatized Word
  frankenstein   frankenstein    frankenstein
        modern         modern          modern
    prometheus      prometheu      prometheus
          mary           mari            mary
wollstonecraft wollstonecraft  wollstonecraft
       shelley        shelley         shelley
       project        project         project
     gutenberg      gutenberg       gutenberg
       edition           edit         edition
           nlp            nlp             nlp
    assignment         assign      assignment
       chapter        chapter         chapter
          best           best            best
         times           time            time
         worst          worst           worst
         times           time            time
           age            age             age
        wisdom         wisdom          wisdom
           age            age             age
   foolishness        foolish  

In [16]:

summary = pd.DataFrame({
    'Processing Stage'      : [
        'Raw tokens',
        'After stop word removal',
        'After stemming',
        'After lemmatization'
    ],
    'Word Count': [
        len(tokens),
        len(valid_words),
        len(stemmed_words),
        len(lemmatized_words)
    ]
})
print("── Preprocessing Summary ──")
print(summary.to_string(index=False))

── Preprocessing Summary ──
       Processing Stage  Word Count
             Raw tokens       41583
After stop word removal       20905
         After stemming       20905
    After lemmatization       20905


---
## Q1(c) — Feature Extraction

In [17]:

sample_words_ohe = list(dict.fromkeys(valid_words))[:30] 

lb = LabelBinarizer()
ohe_matrix = lb.fit_transform(sample_words_ohe)


ohe_df = pd.DataFrame(
    ohe_matrix,
    index=sample_words_ohe,
    columns=lb.classes_
)

print("── One Hot Encoding Output ──")
print(f"Shape : {ohe_df.shape}  (words × unique_classes)")
print()

print(ohe_df.iloc[:10, :15].to_string())
print("\n[Showing 10 words × 15 columns for readability — full matrix:", ohe_df.shape, "]") 

── One Hot Encoding Output ──
Shape : (30, 30)  (words × unique_classes)

                age  assignment  belief  best  chapter  darkness  despair  edition  epoch  everything  foolishness  frankenstein  gutenberg  hope  incredulity
frankenstein      0           0       0     0        0         0        0        0      0           0            0             1          0     0            0
modern            0           0       0     0        0         0        0        0      0           0            0             0          0     0            0
prometheus        0           0       0     0        0         0        0        0      0           0            0             0          0     0            0
mary              0           0       0     0        0         0        0        0      0           0            0             0          0     0            0
wollstonecraft    0           0       0     0        0         0        0        0      0           0            0             0   

In [ ]:

page_docs = []
for page_text in all_text_pages:
    
    t = page_text.lower()
    t = re.sub(r'\d+', '', t)
    t = re.sub(r'[^a-z\s]', '', t)
    t = re.sub(r'\s+', ' ', t).strip()
    t = t.translate(str.maketrans('', '', string.punctuation))
    words = [w for w in t.split() if w not in STOP_WORDS and len(w) > 2]
    if words:
        page_docs.append(' '.join(words))

print(f"📄 Total non-empty page documents for TF-IDF: {len(page_docs)}")


tfidf_vectorizer = TfidfVectorizer(
    max_features=50,    
    min_df=2,           
    max_df=0.85         
)
tfidf_matrix = tfidf_vectorizer.fit_transform(page_docs)

feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"\n📌 TF-IDF Feature Names ({len(feature_names)} terms):")
print(list(feature_names))

📄 Total non-empty page documents for TF-IDF: 110

📌 TF-IDF Feature Names (50 terms):
['black', 'cold', 'could', 'data', 'direct', 'document', 'entering', 'epoch', 'every', 'everything', 'families', 'family', 'find', 'first', 'form', 'foundations', 'frequency', 'funeral', 'gate', 'going', 'hobbit', 'hole', 'human', 'husband', 'intelligence', 'laid', 'language', 'learning', 'lemma', 'like', 'little', 'lodge', 'man', 'mother', 'natural', 'nothing', 'process', 'processing', 'season', 'sky', 'stemming', 'text', 'times', 'understanding', 'upon', 'way', 'whenever', 'white', 'word', 'yesterday']


In [ ]:

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names
)
tfidf_df.index.name = 'Page'

print("── TF-IDF Matrix (first 10 pages × all features) ──")
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 150)
pd.set_option('display.float_format', '{:.4f}'.format)
print(tfidf_df.head(10).to_string())
print(f"\nFull TF-IDF Matrix Shape: {tfidf_df.shape}  (pages × features)")

── TF-IDF Matrix (first 10 pages × all features) ──
      black   cold  could   data  direct  document  entering  epoch  every  everything  families  family   find  first   form  foundations  frequency  funeral   gate  going  hobbit   hole  human  husband  intelligence   laid  language  learning  lemma   like  little  lodge    man  mother  natural  nothing  process  processing  season    sky  stemming   text  times  understanding   upon    way  whenever  white   word  yesterday
Page                                                                                                                                                                                                                                                                                                                                                                                                                                          
0    0.0000 0.0000 0.0000 0.0000  0.0000    0.0000    0.0000 0.0000 0.0000      0.0000

In [ ]:

mean_tfidf = tfidf_df.mean(axis=0).sort_values(ascending=False)

mean_tfidf_df = pd.DataFrame({
    'Term'      : mean_tfidf.index,
    'Mean_TF_IDF': mean_tfidf.values
}).reset_index(drop=True)

print("── Top 20 Terms by Mean TF-IDF Score ──")
print(mean_tfidf_df.head(20).to_string(index=False))

── Top 20 Terms by Mean TF-IDF Score ──
     Term  Mean_TF_IDF
     word       0.1905
     gate       0.1295
 language       0.1283
     hole       0.1206
   hobbit       0.1206
 stemming       0.1134
      man       0.1098
   little       0.1094
 whenever       0.1094
 learning       0.1018
  natural       0.0935
frequency       0.0911
 document       0.0911
    black       0.0863
    lodge       0.0863
    could       0.0840
     data       0.0837
   family       0.0834
  husband       0.0834
      way       0.0823


---
## Q1(d) — TF-IDF Scatter Plot Using Plotly

In [ ]:

fig = go.Figure()


fig.add_trace(go.Scatter(
    x=mean_tfidf_df['Term'],
    y=mean_tfidf_df['Mean_TF_IDF'],
    mode='markers+text',
    text=mean_tfidf_df['Term'],
    textposition='top center',
    marker=dict(
        size=mean_tfidf_df['Mean_TF_IDF'] * 1000 + 8,   
        color=mean_tfidf_df['Mean_TF_IDF'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='TF-IDF Score'),
        line=dict(width=1, color='DarkSlateGrey')
    ),
    hovertemplate='<b>%{x}</b><br>TF-IDF Score: %{y:.5f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='📊 TF-IDF Scores of Top Terms — Frankenstein (Mary Shelley)',
        font=dict(size=18, color='#2C3E50'),
        x=0.5
    ),
    xaxis=dict(
        title='Terms (Words)',
        tickangle=-40,
        tickfont=dict(size=11)
    ),
    yaxis=dict(
        title='Mean TF-IDF Score',
        tickfont=dict(size=11)
    ),
    plot_bgcolor='#F7F9FC',
    paper_bgcolor='white',
    height=600,
    margin=dict(l=60, r=60, t=90, b=120),
    showlegend=False
)

fig.update_xaxes(showgrid=True, gridcolor='#E8EAF0')
fig.update_yaxes(showgrid=True, gridcolor='#E8EAF0')

fig.show()
print("✅ Plotly scatter plot rendered successfully!")

✅ Plotly scatter plot rendered successfully!


In [ ]:

top15 = mean_tfidf_df.head(15)

fig2 = px.bar(
    top15,
    x='Term',
    y='Mean_TF_IDF',
    color='Mean_TF_IDF',
    color_continuous_scale='Plasma',
    title='🏆 Top 15 Most Important Terms by Mean TF-IDF Score',
    labels={'Mean_TF_IDF': 'Mean TF-IDF Score', 'Term': 'Words / Terms'},
    text=top15['Mean_TF_IDF'].round(5)
)

fig2.update_layout(
    title_x=0.5,
    plot_bgcolor='#F7F9FC',
    paper_bgcolor='white',
    height=500,
    xaxis_tickangle=-30
)
fig2.update_traces(textposition='outside')
fig2.show()
print(" Bar chart rendered successfully!")

 Bar chart rendered successfully!


---
## 📋 Final Summary Report

In [23]:
print("="*60)
print("        COMPLETE NLP PIPELINE SUMMARY")
print("="*60)
print(f"  PDF File             : Frankenstein (Mary Shelley)")
print(f"  Total Pages          : {total_pages}")
print(f"  Total Characters     : {len(full_text):,}")
print(f"  Raw Token Count      : {len(tokens):,}")
print(f"  Stop Words Removed   : {len(stopwords_found):,}")
print(f"  Valid Words          : {len(valid_words):,}")
print(f"  Stemmed Words        : {len(stemmed_words):,}")
print(f"  Lemmatized Words     : {len(lemmatized_words):,}")
print(f"  OHE Shape            : {ohe_df.shape}")
print(f"  TF-IDF Matrix Shape  : {tfidf_df.shape}")
print(f"  TF-IDF Features      : {len(feature_names)}")
print("="*60)
print("  Regex Patterns Used:")
print(f"    Numbers  → {REGEX_NUMBERS}")
print(f"    Special  → {REGEX_SPECIAL}")
print(f"    Spaces   → {REGEX_SPACES}")
print("="*60)
print("  Visualization  : Plotly Scatter Plot + Bar Chart")
print("="*60)

        COMPLETE NLP PIPELINE SUMMARY
  PDF File             : Frankenstein (Mary Shelley)
  Total Pages          : 110
  Total Characters     : 234,239
  Raw Token Count      : 41,583
  Stop Words Removed   : 20,678
  Valid Words          : 20,905
  Stemmed Words        : 20,905
  Lemmatized Words     : 20,905
  OHE Shape            : (30, 30)
  TF-IDF Matrix Shape  : (110, 50)
  TF-IDF Features      : 50
  Regex Patterns Used:
    Numbers  → \d+
    Special  → [^a-z\s]
    Spaces   → \s+
  Visualization  : Plotly Scatter Plot + Bar Chart
